# Autopsy of a wrong verdict

This month an automated screen tested target encoding on `playground-series-s6e9` -
668,665 rows, predict who buys an electric vehicle, scored by
roc_auc - and recorded a verdict: **harmful**, -0.001215
against an accept bar of 0.000153.

The verdict was wrong. Target encoding was worth about **+0.003** in
cross-validation and **+0.0037** on the public leaderboard: the largest single
gain measured all month, on a board where the top hundred entries sit within a
few thousandths of each other.

Nothing was wrong with the folds, the model, or the metric. There is a
well-known notebook genre about CV gains that turn out to be noise. This is the
opposite failure and it is worse, because a false rejection never embarrasses
you later. It just quietly costs you the gain and you never find out.

Three protocol details did it, and none of them is exotic.

1. An eligibility rule - encode only columns with at least 50 rows per distinct
   value - was evaluated **inside each CV fold**, on 4/5 of the data. `Annual_Income_USD`
   has 50.6 rows per level on the full training set and
   40.5 inside a fold. The one column carrying the entire effect
   was dropped before the experiment began.
2. The arm bundled every eligible column into a single delta. Eleven columns
   worth nothing, plus one excluded winner, averages to nothing.
3. The "safer" variant restricted encoding to low-cardinality columns - which
   are precisely the columns a tree already handles.

Every failure below is reproduced by a cell you can run.

## Setup

LightGBM and a deliberately modest configuration, so this runs quickly enough to fork and rerun.

In [ ]:
import glob, os
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

# Find the data rather than hardcoding a path. Kaggle has moved competition data
# between /kaggle/input/<slug>/ and /kaggle/input/competitions/<slug>/, and a
# notebook that hardcodes either one breaks without warning when it changes.
found = sorted(glob.glob("/kaggle/input/**/*train*.csv", recursive=True))
if not found:
    raise FileNotFoundError(
        "no training csv under /kaggle/input - attach the competition data via "
        "'+ Add Input' in the sidebar")
DATA, TRAIN = os.path.dirname(found[0]), found[0]
TARGET, ID = "Will_Buy_EV", "id"
print("data:", TRAIN)

train = pd.read_csv(TRAIN)

# The target is usually numeric, but not always. Everything below takes means of
# it, so encode once here. Sorted rather than first-seen, so the mapping is a
# property of the data and not of the row order.
CLASSES = None
if not pd.api.types.is_numeric_dtype(train[TARGET]):
    CLASSES = sorted(train[TARGET].dropna().unique().tolist(), key=str)
    train[TARGET] = train[TARGET].map({c: i for i, c in enumerate(CLASSES)})
    print(f"target was categorical: {CLASSES} -> 0..{len(CLASSES) - 1}")

print(f"{train.shape[0]:,} rows x {train.shape[1]} columns")
train.head(3)

In [ ]:
FEATURES = [c for c in train.columns if c not in (TARGET, ID)]
CAT = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(train[c])]

def prep(df):
    X = df[FEATURES].copy()
    for c in CAT:
        X[c] = X[c].fillna("__MISSING__").astype("category")
    return X

def score(y, p):
    return roc_auc_score(y, p)

METRIC = "roc_auc"
print(f"{len(FEATURES)} features, {len(CAT)} categorical, metric {METRIC}")

In [ ]:
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,
              min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8,
              bagging_freq=1, verbose=-1, n_jobs=-1)
N_TREES = 400

def cv_score(X, y, seed, n_splits=5):
    """One full cross-validation under seed `seed` - partition AND model.

    The seed moves the PARTITION and the model's own randomness together. Pinning
    the model seed and varying only the partition gives a floor about ten times
    too small - it measures one source of run-to-run variation and calls it all of
    them. A bar built that way then marks trivial deltas as real wins.
    """
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for i, (tr, va) in enumerate(kf.split(X, y)):
        m = lgb.train(dict(PARAMS, seed=seed * 100 + i),
                      lgb.Dataset(X.iloc[tr], y[tr]), num_boost_round=N_TREES)
        oof[va] = m.predict(X.iloc[va])
    return score(y, oof), oof

y = train[TARGET].values
X = prep(train)

## The bar

Fold-to-fold spread is not your uncertainty. It tells you how much the score
moves across *subsets of your data*. It says almost nothing about how much the
fold **mean** would move if you had split the data differently - and that second
quantity is the one an idea has to beat.

So measure it directly: run the entire cross-validation again under a different
seed - moving the partition **and** the model's own randomness together, because
both vary between two runs of your pipeline - several times, and take the
standard deviation of the means.

Three seeds gives a rough estimate: the std of a std from three samples carries
roughly 50% relative error, so treat the bar as an order of magnitude, not a
threshold to three significant figures.

In [ ]:
means = []
for s in [0, 1, 2]:
    m, _ = cv_score(X, y, seed=s)
    means.append(m)
    print(f"partition seed {s}: {m:.6f}")

floor = float(np.std(means))
BAR = 2 * floor
print(f"\nnoise floor (std of the means) = {floor:.6f}")
print(f"accept bar (2x)                = {BAR:.6f}")

## 1. The threshold, measured both ways

The eligibility statistic is rows per distinct value. It reads the feature column
and **never the target**, so it carries no leakage risk - but it was being
computed on the fold's training split rather than on the dataset.

A five-fold split keeps 4/5 of the rows, so every column's rows-per-level shrinks
by 20%. A threshold of T is therefore really a threshold of T/0.8 on the numbers
you would compute on the full data.

In [ ]:
BAR_RPL = 50
rows = []
for c in FEATURES:
    nun = train[c].astype("string").nunique()
    full = len(train) / max(nun, 1)
    fold = (len(train) * 0.8) / max(nun, 1)
    rows.append({"column": c, "levels": nun, "rpl_full": full,
                 "rpl_in_fold": fold,
                 "passes_full": full >= BAR_RPL,
                 "passes_in_fold": fold >= BAR_RPL})
el = pd.DataFrame(rows).sort_values("rpl_full")
print(el.to_string(index=False, formatters={
    "levels": lambda v: f"{v:,}", "rpl_full": lambda v: f"{v:,.1f}",
    "rpl_in_fold": lambda v: f"{v:,.1f}"}))

flip = el[el.passes_full != el.passes_in_fold]
print(f"\ncolumns whose eligibility is decided by the fold split, not the data:")
for _, r in flip.iterrows():
    print(f"  {r['column']}  {r['rpl_full']:.1f} on full data, "
          f"{r['rpl_in_fold']:.1f} in-fold, threshold {BAR_RPL}")
print(f"\neffective threshold a 5-fold screen applies to full-data values: "
      f"{BAR_RPL / 0.8:.1f}")

## 2. The bundle

The screen encoded every eligible column at once and reported one number. Here is
what that number is an average of. Same folds, same model, one variable: which
column is encoded.

In [ ]:
C = "Annual_Income_USD"

def enc_map(keys, yv, prior, smoothing=20.0):
    gg = pd.DataFrame({"k": keys, "y": yv})
    agg = gg.groupby("k")["y"].agg(["sum", "count"])
    return (agg["sum"] + prior * smoothing) / (agg["count"] + smoothing)

def cv_te(cols, seed=0, n_splits=5, inner=5):
    oof = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in skf.split(X, y):
        Xtr, Xva = X.iloc[tr].copy(), X.iloc[va].copy()
        ytr = y[tr]; prior = float(ytr.mean())
        for c in cols:
            keys = train[c].astype("string").fillna("__MISSING__").to_numpy()
            ktr, kva = keys[tr], keys[va]
            full = enc_map(ktr, ytr, prior)
            Xva[f"te_{c}"] = pd.Series(kva).map(full).astype(float).fillna(prior).values
            colv = np.full(len(ktr), prior, dtype=float)
            ikf = StratifiedKFold(n_splits=inner, shuffle=True, random_state=seed)
            for itr, iva in ikf.split(ktr, ytr):
                m = enc_map(ktr[itr], ytr[itr], prior)
                colv[iva] = pd.Series(ktr[iva]).map(m).astype(float).fillna(prior).values
            Xtr[f"te_{c}"] = colv
        mdl = lgb.train(dict(PARAMS, seed=0), lgb.Dataset(Xtr, ytr),
                        num_boost_round=N_TREES)
        oof[va] = mdl.predict(Xva)
    return score(y, oof)

In [ ]:
# The contrast must be another column with MORE levels than the model's 255
# histogram bins - not whichever column tops a ranking, which is a 2-level flag
# a range split reaches for free.
_lv = {c: train[c].astype("string").nunique() for c in FEATURES}
_hi = sorted([c for c in FEATURES if _lv[c] > 255 and c != C],
             key=lambda c: -_lv[c])
RUNNER_UP = _hi[0] if _hi else max(
    (c for c in FEATURES if c != C), key=lambda c: _lv[c])
print(f"contrast column: {RUNNER_UP} ({_lv[RUNNER_UP]:,} levels) "
      f"vs {C} ({_lv[C]:,} levels)")
arms = [("A  baseline, no encoding", []),
        (f"B  encode {C}", [C]),
        (f"C  encode {RUNNER_UP}", [RUNNER_UP])]
res = {}
for name, cols in arms:
    res[name] = cv_te(cols)
    print(f"{name:34s} {METRIC}={res[name]:.6f}")

base = res["A  baseline, no encoding"]
idea_score = res[f"B  encode {C}"]
print()
print(f"{'arm':34s}{'score':>11}{'delta':>12}   vs bar {BAR:.6f}")
for name, s in res.items():
    d = s - base
    v = "-" if name.startswith("A") else ("WIN" if d > BAR else
        "HARMFUL" if d < -BAR else "inside the noise")
    print(f"{name:34s}{s:>11.6f}{d:>+12.6f}   {v}")

## 3. Why the bundle hides it

One column at +0.003 and eleven at zero, averaged into a single arm, is
indistinguishable from nothing - and with the winner excluded by the eligibility
rule, the bundle contained only the zeros. A screen that reports one delta per
*idea* cannot see this. A screen that reports one delta per *column* can.

That is the whole cost of bundling, and it is not specific to target encoding.
Any ablation that groups several features under one name has the same blind spot.

## What to take from it

Three changes, each cheap.

**Decompose any bundled arm that lands near your bar**, in either direction. One
arm per column, same folds. Here it cost about twenty minutes and turned
"harmful" into the largest win of the month.

**Audit every threshold your pipeline evaluates inside a fold.** Any eligibility
statistic that scales with row count - rows per level, minimum category size,
minimum samples per leaf - is effectively tested at `threshold / 0.8` under
five-fold CV. Print each column's full-data value beside its in-fold value, as
the first cell above does. A column that crosses your threshold between the two
is being decided by fold arithmetic rather than by data. If the statistic reads
no target information, compute it once on the full training set: that is not
leakage, and it matches what your submission model will actually see.

**Store verdicts per dataset, not per technique.** The decimal-digit idea was a
clear win on the previous competition in this series and measured null here. A
note that says a technique "works" is stale the day the generator changes.